In [1]:
import polars as pl
from pathlib import Path

In [19]:
DATA_GENERAL = Path("../data_general")
DATA_PERSONAL = Path("../data_personal/Spotify Extended Streaming History")
OUT_DATA = Path('../output') 

In [3]:
general_data_frames = []
for num in range (10) :
    general_data_frames.append(pl.read_parquet(DATA_GENERAL / f'spotify_audio_features_{num}.parquet'))

In [4]:
display(general_data_frames[0].head())

id,name,popularity,null_response,duration_ms,time_signature,key,mode,tempo,danceability,energy,loudness,speechiness,acousticness,instrumentalness,liveness,valence
str,str,i64,i64,i64,i64,i64,i64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""2Pe9cbhOTvOUTDE4bl7zzl""","""I dreamt you died""",0,0,630506,4,6,0,87.683,0.279,0.391,-12.054,0.32,0.816,0.737,0.177,0.0299
"""0wP732NKm8XgXu78XLRWoR""","""It's Death""",0,0,97216,4,5,1,105.298,0.429,0.318,-11.685,0.0566,0.587,0.782,0.202,0.36
"""22L6EJdnjx8oIo7GiF9hLe""","""Preliminary""",0,0,75180,4,0,1,117.657,0.283,0.581,-9.42,0.0555,0.923,0.939,0.106,0.0362
"""3a519lgQ13JXNi0G73mwMT""","""Disparage""",0,0,149447,4,5,0,100.685,0.244,0.995,-0.69,0.125,0.78,0.799,0.132,0.0634
"""27yP7p2lxWYTtnldRN8Kzx""","""Cut Down""",0,0,120816,4,7,1,123.499,0.313,0.618,0.411,0.073,0.843,0.109,0.126,0.187


In [5]:
personal_data_frames = {}
for num in range (2022,2027) :
    personal_data_frames[num] = pl.read_json(
        DATA_PERSONAL / f'Streaming_History_Audio_{num}.json',
        infer_schema_length=None  
        )

In [6]:
personal_data_frames[2022] =personal_data_frames[2022].with_columns(pl.col("spotify_track_uri").str.slice(14,22))

In [7]:
check_mus_id = personal_data_frames[2022]["spotify_track_uri"][0]
print(check_mus_id)

4u7EnebtmKWzUH433cf5Qv


In [8]:
#print(len(general_data_frames))
for g in range(10):
    general_data_frames[g] = general_data_frames[g].rename({ 'id' : 'spotify_track_uri' })
print(general_data_frames[0])

shape: (25_558_893, 17)
┌────────────┬────────────┬───────────┬───────────┬───┬───────────┬───────────┬──────────┬─────────┐
│ spotify_tr ┆ name       ┆ popularit ┆ null_resp ┆ … ┆ acousticn ┆ instrumen ┆ liveness ┆ valence │
│ ack_uri    ┆ ---        ┆ y         ┆ onse      ┆   ┆ ess       ┆ talness   ┆ ---      ┆ ---     │
│ ---        ┆ str        ┆ ---       ┆ ---       ┆   ┆ ---       ┆ ---       ┆ f64      ┆ f64     │
│ str        ┆            ┆ i64       ┆ i64       ┆   ┆ f64       ┆ f64       ┆          ┆         │
╞════════════╪════════════╪═══════════╪═══════════╪═══╪═══════════╪═══════════╪══════════╪═════════╡
│ 2Pe9cbhOTv ┆ I dreamt   ┆ 0         ┆ 0         ┆ … ┆ 0.816     ┆ 0.737     ┆ 0.177    ┆ 0.0299  │
│ OUTDE4bl7z ┆ you died   ┆           ┆           ┆   ┆           ┆           ┆          ┆         │
│ zl         ┆            ┆           ┆           ┆   ┆           ┆           ┆          ┆         │
│ 0wP732NKm8 ┆ It's Death ┆ 0         ┆ 0         ┆ … ┆ 0.587     ┆

In [9]:
status = False
mus_name = ''
for g in range(10):
    for url in general_data_frames[g]['spotify_track_uri']:
        if url == check_mus_id:
            status = True
            mus_name = general_data_frames[g].filter(pl.col('spotify_track_uri') == url).select(['name',])
            break
    if status :
        break

print(status, mus_name)

True shape: (1, 1)
┌─────────────────────────────────┐
│ name                            │
│ ---                             │
│ str                             │
╞═════════════════════════════════╡
│ Bohemian Rhapsody - Remastered… │
└─────────────────────────────────┘


# NOW LET`S FIND OUT TRACK

In [10]:
song_uri = '7CssMew7KGzViJwlKvrIb3'

In [11]:
print(song_uri)
for g in range (10):
    if song_uri in general_data_frames[g]['spotify_track_uri']:
        print(g)
        stats = general_data_frames[g].filter(pl.col('spotify_track_uri') == song_uri)
        break


7CssMew7KGzViJwlKvrIb3
9


In [12]:
display(stats)

spotify_track_uri,name,popularity,null_response,duration_ms,time_signature,key,mode,tempo,danceability,energy,loudness,speechiness,acousticness,instrumentalness,liveness,valence
str,str,i64,i64,i64,i64,i64,i64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""7CssMew7KGzViJwlKvrIb3""","""Беспонтовый пирожок""",42,0,191114,4,11,0,121.504,0.562,0.598,-9.105,0.0304,0.345,0.222,0.408,0.924


# I have priority on the  energy, instrumentalness and valence

In [13]:
# Writ None in coeficients if you want to not search for this parameters
# In cof`s writre cof like 

cof_tempo = None        #       diff in BPM
cof_dance = 0.2         #       from 0 to 1
cof_energy = 0.2       #       from 0 to 1
cof_loud = None         #       diff in dB (0 dB - max loudness)
cof_speech = None       #       from 0 to 1
cof_acoustic = 0.4     #       from 0 to 1
cof_instr = 0.4        #       from 0 to 1 
cof_live = 0.2         #       from 0 to 1 
cof_valence = 0.1      #       from 0 to 1 


#energy = (float(stats['energy'][0]) - 0.02 , float(stats['energy'][0]) + 0.02 )
#instr = (float(stats['instrumentalness'][0]) - 0.02 , float(stats['instrumentalness'][0]) + 0.02 )
#valence = (float(stats['valence'][0]) - 0.02 , float(stats['valence'][0]) + 0.02 )

#print(energy)

In [14]:
result = general_data_frames[9]#

In [15]:
if cof_tempo:
    ...
if cof_dance:
    state = stats['danceability']
    result = result.filter((pl.col('danceability').is_between(state*(1 - cof_dance) ,state*(1 + cof_dance) )))
if cof_energy:
    state = stats['energy']
    result = result.filter((pl.col('energy').is_between(state*(1 - cof_energy) ,state*(1 + cof_energy) )))
if cof_loud:
    ...
if cof_speech:
    state = stats['speechiness']
    result = result.filter((pl.col('speechiness').is_between(state*(1 - cof_speech) ,state*(1 + cof_speech) )))
if cof_acoustic:
    state = stats['acousticness']
    result = result.filter((pl.col('acousticness').is_between(state*(1 - cof_acoustic) ,state*(1 + cof_acoustic) )))
if cof_instr:
    state = stats['instrumentalness']
    result = result.filter((pl.col('instrumentalness').is_between(state*(1 - cof_instr) ,state*(1 + cof_instr) )))
if cof_live:
    state = stats['liveness']
    result = result.filter((pl.col('liveness').is_between(state*(1 - cof_live) ,state*(1 + cof_live) )))
if cof_valence:
    state = stats['valence']
    result = result.filter((pl.col('valence').is_between(state*(1 - cof_valence) ,state*(1 + cof_valence) )))

In [16]:
print(len(result))
result = result.with_columns(
    pl.format("spotify:track:{}", pl.col("spotify_track_uri")).alias("spotify_track_uri")
)
display(result)


180


spotify_track_uri,name,popularity,null_response,duration_ms,time_signature,key,mode,tempo,danceability,energy,loudness,speechiness,acousticness,instrumentalness,liveness,valence
str,str,i64,i64,i64,i64,i64,i64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""spotify:track:75DcHyFjGLcfr70v…","""Skitter Dis""",0,0,265000,4,5,0,84.001,0.581,0.663,-14.39,0.0481,0.281,0.223,0.429,0.955
"""spotify:track:0hezsdUkzRewn6IB…","""Mountain Harpies""",0,0,63997,4,0,1,112.515,0.541,0.528,-9.562,0.037,0.397,0.206,0.397,0.871
"""spotify:track:7D3edsf7p3kAk1rJ…","""Don't Get Lost""",0,0,97143,1,8,0,74.099,0.593,0.634,-6.844,0.944,0.447,0.309,0.45,0.834
"""spotify:track:5SHmQtxpjGppCYwh…","""hoodie szn""",0,0,202320,4,11,1,144.445,0.59,0.667,-7.514,0.0595,0.455,0.228,0.433,0.902
"""spotify:track:2Pt0lekAdS2dpQ6d…","""Vuki Vuki Vuh (Speed Up)""",0,0,88685,4,1,0,76.823,0.489,0.661,-4.883,0.0941,0.393,0.179,0.478,0.869
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""spotify:track:78DtWOYRPmW3tQF6…","""Plaza Muse""",0,0,144472,4,5,0,108.252,0.563,0.537,-11.692,0.0291,0.211,0.162,0.362,0.918
"""spotify:track:5ewkAUmYKfQBom7p…","""Festive Day""",0,0,188720,4,0,1,123.977,0.61,0.578,-10.159,0.0355,0.403,0.308,0.378,0.851
"""spotify:track:5zR5dkCGtZRvm1DC…","""Wish Meteor""",0,0,143613,3,0,1,93.617,0.647,0.696,-10.645,0.0599,0.252,0.247,0.376,0.845


In [30]:
file_path = OUT_DATA / f"Like '{stats['name'][0]}'.csv"
result.select('spotify_track_uri').write_csv(file_path , separator=',')